### Data Ingestion

In [3]:
### document datastructure

from langchain_core.documents import Document

In [2]:
doc = Document(
    page_content="This is the content of the document.",
    metadata={
    "source": "example.txt", 
    "author": "Shubhradeep Roy", 
    "date_created": "2024-03-29"
    }
)
doc

NameError: name 'Document' is not defined

### Loading and chunking

In [ ]:
# loading documents
from fastapi import FastAPI, File, UploadFile, Form
from langchain_core.documents import Document
from langchain_community.document_loaders import PyPDFLoader,PyMuPDFLoader,DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter # Used for splitting the documents into smaller chunks
from langchain_experimental.text_splitter import SemanticChunker # Used for splitting the documents into smaller chunks based on semantic meaning

from langchain_community.embeddings import HuggingFaceEmbeddings # Used as an embedding interface to be fed to Semantic Chunker for generating embeddings for the semantic chunking
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any
import os
from pypdf import PdfReader
import io
# dir_loader = DirectoryLoader(
#     "../data/PDF_files",
#     glob="**/*.pdf",
#     loader_cls=PyMuPDFLoader,
#     show_progress=False,
# )
#  pdf_docs = dir_loader.load()

app = FastAPI()

@app.post("/upload")
async def upload(
    file: UploadFile = File(...),
    doc_type: str = Form(...)
):
    content = await file.read()
    if file.filename.endswith(".pdf"):
        pdf = PdfReader(io.BytesIO(content))
        text = ""
        for page in pdf.pages:
            text += page.extract_text() or ""
    else:
       text = content.decode("utf-8", errors="ignore")

    pdf_docs = [
        Document(
            page_content=text,
            metadata={"source": file.filename, "type": doc_type}
        )
    ]


    # Recursive Chunking i.e chunking based on separators and chunk size
    def split_docs(docs, chunk_size=1000, chunk_overlap=200):
        text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, 
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
        )
        chunks =text_splitter.split_documents(docs)
        print(f"Total number of chunks produced: {len(chunks)} from {len(docs)} documents.")
        return chunks

# Pre-splitting before semantic chunking
    pre_chunks = split_docs(pdf_docs, chunk_size=2000, chunk_overlap=200)

# Semantic Chunking i.e chunking based on semantic meaning of the text by generating embeddings and then clustering them
    embedding_interface = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

    def semantic_chunking(docs):
       text_splitter = SemanticChunker(
        embedding_interface,
        breakpoint_threshold_type="percentile",
        breakpoint_threshold_amount= 90
       )
       chunks = text_splitter.split_documents(docs)
       print(f"Total number of chunks produced: {len(chunks)} from {len(docs)} documents.")
       return chunks
    semantic_chunks = semantic_chunking(pre_chunks)
    # print(f"Sample chunk: {semantic_chunks[0]}")
    # print(f"One more sample chunk: {semantic_chunks[225]}")

    

    class EmbeddingManager:
       def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """ Hugging face model is used over here """
        self.model = None
        self.model_name = model_name
        self._load_model()

       def _load_model(self):
          """ Load the sentence transformer model for generating embeddings. """
          try:
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully... Embedding dimension : {self.model.get_sentence_embedding_dimension()}")
          except Exception as e:
            print(f"Error loading model: {e}")
            raise e
        
       def generate_embeddings(self, texts: List[str]) -> np.ndarray:
         """ Generate embeddings for a list of texts. """
         try:
           embeddings = self.model.encode(texts, show_progress_bar=True)
           print(f"Generated embeddings with shape: {embeddings.shape}")
           return embeddings
         except Exception as e:
           print(f"Error generating embeddings: {e}")
           raise e

    

    class VectorStore:
      def __init__(self, collection_name, persist_directory: str = "../data/chromadb"):
        """Initialize the ChromaDB client and create a collection for storing document embeddings."""
        self.client = None
        self.collection = None
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self._initialize_chromadb()

      def _initialize_chromadb(self):
        """Initialize the ChromaDB client and create a collection for storing document embeddings."""
        try:
            os.makedirs(self.persist_directory, exist_ok=True)

            self.client = chromadb.PersistentClient(path=self.persist_directory)

            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={f"description": "Collection for storing {self.collection_name} embeddings"},
            )

            print(f"ChromaDB collection '{self.collection_name}' initialized successfully.")
        except Exception as e:
            print(f"Error initializing ChromaDB: {e}")
            raise e
    
      def get_collection_info(self):
        """Get information about the ChromaDB collection."""
        
        print(f"Collection count: {self.collection.count()}")
        collections = self.client.list_collections()

        for col in collections:
           print(col.name, col.id)


      def add_documents(self, documents: List[Any], embeddings: np.ndarray): 
        """
        Add docs and tehir corresponding embeddings to the ChromaDB collection.
        Args:
            documents (List[Any]): List of documents to be added.
            embeddings (np.ndarray): Corresponding embeddings for the documents.
        """   

        if(len(documents) != len(embeddings)):
            raise ValueError("The number of documents and embeddings must be the same.")
        print(f"Adding {len(documents)} documents to the vector store...")

        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            # Preparing the metadata
            metadata = dict(doc.metadata)  # Copying the existing metadata from the document
            metadata['doc_idx'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)

            # Document content
            documents_text.append(doc.page_content)

            # Embedding list
            embeddings_list.append(embedding.tolist())  # Converting numpy array to list for JSON serialization
         
        # Adding documents to the ChromaDB collection
        try:
            self.collection.add(
                ids=ids,
                documents=documents_text,
                metadatas=metadatas,
                embeddings=embeddings_list
            )
            print(f"Successfully added {len(documents)} documents to the vector store.")
            print(f"Total documents in the collection: {self.collection.count()}")
        
        except Exception as e:
            print(f"Error adding documents to the vector store: {e}")
            raise e
        
    vector_store = VectorStore(f"quested_{doc_type.lower()}")# pass the collection name as an argument while creating the instance of the VectorStore class
    texts = [doc.page_content for doc in semantic_chunks]
    embedding_manager = EmbeddingManager()

    embeddings = embedding_manager.generate_embeddings(texts)

    vector_store.add_documents(semantic_chunks, embeddings)

    return {
         "message": "Uploaded successfully",
         "collection": vector_store.collection_name,
         "chunks_stored": len(semantic_chunks)
     }


Total number of chunks produced: 332 from 300 documents.


C:\Users\Shubhradeep Roy\AppData\Local\Temp\ipykernel_19788\550428161.py:37: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_interface = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3669.12it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Total number of chunks produced: 1169 from 332 documents.
Sample chunk: page_content='Competitive Programmer’s Handbook
Antti Laaksonen
Draft August 19, 2019' metadata={'producer': 'pdfTeX-1.40.18', 'creator': 'LaTeX with hyperref package', 'creationdate': '2019-08-19T21:35:09+03:00', 'source': 'PDF_files', 'file_path': '..\\data\\PDF_files\\Sample.pdf', 'total_pages': 300, 'format': 'PDF 1.5', 'title': '', 'author': 'Shubhradeep Roy', 'subject': 'Competitive Programming', 'keywords': '', 'moddate': '2019-08-19T21:35:09+03:00', 'trapped': '', 'modDate': "D:20190819213509+03'00'", 'creationDate': "D:20190819213509+03'00'", 'page': 0, 'date_created': '2024-03-29'}
One more sample chunk: page_content='ii' metadata={'producer': 'pdfTeX-1.40.18', 'creator': 'LaTeX with hyperref package', 'creationdate': '2019-08-19T21:35:09+03:00', 'source': 'PDF_files', 'file_path': '..\\data\\PDF_files\\Sample.pdf', 'total_pages': 300, 'format': 'PDF 1.5', 'title': '', 'author': 'Shubhradeep Roy', 'subjec

### Embedding and VectoreStoreDB

In [4]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity #Used for performing similarity search on the embeddings of the documents



c:\Users\Shubhradeep Roy\Documents\WebDev\RAG_for_Practice\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any

class EmbeddingManager:
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """ Hugging face model is used over here """
        self.model = None
        self.model_name = model_name
        self._load_model()

    def _load_model(self):
        """ Load the sentence transformer model for generating embeddings. """
        try:
          self.model = SentenceTransformer(self.model_name)
          print(f"Model loaded successfully... Embedding dimension : {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
          print(f"Error loading model: {e}")
          raise e
        
    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """ Generate embeddings for a list of texts. """
        try:
          embeddings = self.model.encode(texts, show_progress_bar=True)
          print(f"Generated embeddings with shape: {embeddings.shape}")
          return embeddings
        except Exception as e:
          print(f"Error generating embeddings: {e}")
          raise e

embedding_manager = EmbeddingManager()
embedding_manager



c:\Users\Shubhradeep Roy\Documents\WebDev\RAG_for_Practice\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2913.58it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded successfully... Embedding dimension : 384


### Vector Store


In [2]:
import os

class VectorStore:
    def __init__(self, collection_name, persist_directory: str = "../data/chromadb"):
        """Initialize the ChromaDB client and create a collection for storing document embeddings."""
        self.client = None
        self.collection = None
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self._initialize_chromadb()

    def _initialize_chromadb(self):
        """Initialize the ChromaDB client and create a collection for storing document embeddings."""
        try:
            os.makedirs(self.persist_directory, exist_ok=True)

            self.client = chromadb.PersistentClient(path=self.persist_directory)

            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={f"description": "Collection for storing {self.collection_name} embeddings"},
            )

            print(f"ChromaDB collection '{self.collection_name}' initialized successfully.")
        except Exception as e:
            print(f"Error initializing ChromaDB: {e}")
            raise e
    
    def get_collection_info(self):
        """Get information about the ChromaDB collection."""
        
        print(f"Collection count: {self.collection.count()}")
        collections = self.client.list_collections()

        for col in collections:
           print(col.name, col.id)


    def add_documents(self, documents: List[Any], embeddings: np.ndarray): 
        """
        Add docs and tehir corresponding embeddings to the ChromaDB collection.
        Args:
            documents (List[Any]): List of documents to be added.
            embeddings (np.ndarray): Corresponding embeddings for the documents.
        """   

        if(len(documents) != len(embeddings)):
            raise ValueError("The number of documents and embeddings must be the same.")
        print(f"Adding {len(documents)} documents to the vector store...")

        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            # Preparing the metadata
            metadata = dict(doc.metadata)  # Copying the existing metadata from the document
            metadata['doc_idx'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)

            # Document content
            documents_text.append(doc.page_content)

            # Embedding list
            embeddings_list.append(embedding.tolist())  # Converting numpy array to list for JSON serialization
         
        # Adding documents to the ChromaDB collection
        try:
            self.collection.add(
                ids=ids,
                documents=documents_text,
                metadatas=metadatas,
                embeddings=embeddings_list
            )
            print(f"Successfully added {len(documents)} documents to the vector store.")
            print(f"Total documents in the collection: {self.collection.count()}")
        
        except Exception as e:
            print(f"Error adding documents to the vector store: {e}")
            raise e
        
vector_store = VectorStore("pdf_documents")# pass the collection name as an argument while creating the instance of the VectorStore class
vector_store2 = VectorStore("hello_docs")

vector_store


ChromaDB collection 'pdf_documents' initialized successfully.
ChromaDB collection 'hello_docs' initialized successfully.


### Classes and Functions Usage

In [15]:
texts = [doc.page_content for doc in semantic_chunks]

embeddings = embedding_manager.generate_embeddings(texts)

vector_store.add_documents(semantic_chunks, embeddings)


Batches: 100%|██████████| 37/37 [00:29<00:00,  1.27it/s]


Generated embeddings with shape: (1169, 384)
Adding 1169 documents to the vector store...
Successfully added 1169 documents to the vector store.
Total documents in the collection: 1784


In [3]:
vector_store.get_collection_info()
# collections = vector_store.list_collections()



Collection count: 1784
pdf_documents 3a5bbef8-b21a-4d61-b7bc-958a4a93486b
hello_docs 3aab05cc-58f7-454e-97ce-64e5a91da583


### RAG Retriever pipeline from Vector Store


In [4]:
class RAGRetriever:
    def __init__(self,embedding_manager: EmbeddingManager):
        self.vector_store = None
        self.embedding_manager = embedding_manager

    def retrieve(self, doc_type, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """Retrieve relevant documents based on the query."""
        self.vector_store = VectorStore(doc_type) # Initialize the vector store with the appropriate collection based on the doc_type
        # Generate embedding for the query
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        try:
            print(f"collection name: {self.vector_store.collection.name}")
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k,
            )

            retrieved_docs = []

            if results.get('documents') and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]

                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Converting the distance to similarity score because chromadb uses cosine distance
                    similarity_score = 1 - distance

                    if similarity_score >= score_threshold:  # Filter out documents with zero similarity
                        retrieved_docs.append({
                            "id": doc_id,
                            "content": document,
                            "metadata": metadata,
                            "distance": distance,
                            "similarity_score": similarity_score,
                            "rank": i + 1
                        })

                print(f"Retrieved {len(retrieved_docs)} documents for the query: '{query}'")
            else:
                print(f"No documents found for the query: '{query}'")

            return retrieved_docs
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

RAG = RAGRetriever(embedding_manager)


### Integrating vector db context pipeline with LLM output

In [ ]:
from langchain_groq import ChatGroq
from dotenv import load_dotenv
import os
# Load environment variables from .env file
load_dotenv()

# The client gets the API key from the environment variable `GROQ_API_KEY`.
groq_api_key = os.getenv("GROQ_API_KEY")
llm = ChatGroq(groq_api_key=groq_api_key,model_name = "llama-3.3-70b-versatile",temperature=0.1,max_tokens=1024)



def rag(doc_type, query,retriever,llm,top_k=3):
    ## retrieving the context
    results = retriever.retrieve(doc_type,query,top_k=top_k)
    context = [result['content'] for result in results] if results else ""
    # print(context)
    if not context:
        return "No relevant documents found to answer the requested query."
    
    ## System prompt for the LLM
    system_prompt = f"""You are a helpful assistant for answering queries with respect to a given context.
    Context: {{context}}
    Query: {{query}}
    Answer the query based only on the provided context. Please try to be as concise as possible while answering the query. Do not hallucinate or provide any information which is not present in the provided context.
      NB: If you encounter \n\n or \n in the context, treat them as newline escape sequence, please go to a different line and do not include these characters in the answer."""
    
    response = llm.invoke([system_prompt.format(context=context, query=query)])
    return response.content

query = "Explain adjacency list representation of a graph"
answer = rag('hello_docs',query, RAG, llm,top_k=10)
answer
    

ChromaDB collection 'hello_docs' initialized successfully.


Batches: 100%|██████████| 1/1 [00:00<00:00,  3.14it/s]

Generated embeddings with shape: (1, 384)
collection name: hello_docs
No documents found for the query: 'Explain adjacency list representation of a graph'


'No relevant documents found to answer the requested query.'